# PROMISE Multi-Project Preprocessing

This notebook runs and inspects the generalized preprocessing pipeline implemented in `scripts/preprocess_promise.py`.

The script is the single source of truth. The notebook only wraps execution and validates the generated outputs.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SCRIPT_PATH = REPO_ROOT / 'scripts' / 'preprocess_promise.py'
PROJECTS_ROOT = REPO_ROOT / 'projects'
OUTPUT_ROOT = REPO_ROOT / 'outputs'
COMBINED_CSV = OUTPUT_ROOT / 'promise' / 'promise_preprocessed_standard.csv'
COMBINED_SUMMARY = OUTPUT_ROOT / 'promise' / 'promise_preprocess_summary.json'

print('Repository:', REPO_ROOT)
print('Preprocessing script:', SCRIPT_PATH)
print('Projects root:', PROJECTS_ROOT)

## 1. Discover Available PROMISE Datasets

The preprocessing script auto-discovers CSV files under `projects/*/*.csv`. Each dataset folder is treated as the source-code root, and Java package declarations are used for class-to-source mapping.

In [ ]:
dataset_csvs = sorted(PROJECTS_ROOT.glob('*/*.csv'))
rows = []
for path in dataset_csvs:
    df = pd.read_csv(path)
    if 'bug' not in df.columns:
        continue
    rows.append({
        'dataset_name': path.stem,
        'csv_path': str(path.relative_to(REPO_ROOT)),
        'raw_samples': len(df),
        'raw_defective_samples': int((pd.to_numeric(df['bug'], errors='coerce').fillna(0) > 0).sum()),
    })

discovery_df = pd.DataFrame(rows)
discovery_df

## 2. Run Generalized Preprocessing

This runs the same command you can use from the terminal:

```bash
python scripts/preprocess_promise.py
```

Default behavior:

- binary label: `bug > 0 -> 1`, else `0`
- median imputation for metric columns
- `log1p` transform
- global `StandardScaler` across all discovered projects
- one combined CSV plus per-project CSV and mapping artifacts

In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPT_PATH)],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)

## 3. Combined Output Summary

In [ ]:
summary = json.loads(COMBINED_SUMMARY.read_text())
summary_view = {
    key: summary[key]
    for key in [
        'num_datasets',
        'rows',
        'columns',
        'feature_columns',
        'feature_transform',
        'scaler',
        'scale_scope',
        'defective_samples',
        'non_defective_samples',
        'mapped_classes',
        'unmapped_classes',
        'combined_preprocessed_file',
    ]
}
summary_view

## 4. Load The Combined SDP Dataset

In [ ]:
combined_df = pd.read_csv(COMBINED_CSV)
print('shape:', combined_df.shape)
print('columns:', combined_df.columns.tolist())
combined_df.head()

## 5. Per-Project Mapping and Label Statistics

In [ ]:
project_rows = []
for item in summary['project_summaries']:
    project_rows.append({
        'dataset_name': item['dataset_name'],
        'rows': item['rows'],
        'mapped_classes': item['mapped_classes'],
        'unmapped_classes': item['total_classes'] - item['mapped_classes'],
        'non_defective': item['bug_label_counts'].get('0', 0),
        'defective': item['bug_label_counts'].get('1', 0),
        'match_strategy_counts': item['match_strategy_counts'],
    })

project_summary_df = pd.DataFrame(project_rows)
project_summary_df

## 6. Feature Validation

The combined file should contain identifier columns, 20 metric features, the binary label, source mapping metadata, and no non-finite metric values.

In [ ]:
metric_columns = [
    'wmc', 'dit', 'noc', 'cbo', 'rfc', 'lcom', 'ca', 'ce', 'npm', 'lcom3',
    'loc', 'dam', 'moa', 'mfa', 'cam', 'ic', 'cbm', 'amc', 'max_cc', 'avg_cc'
]

checks = {
    'expected_rows_match_summary': len(combined_df) == summary['rows'],
    'expected_metric_count': len(metric_columns) == summary['feature_columns'],
    'all_metric_columns_present': set(metric_columns).issubset(combined_df.columns),
    'labels_binary': set(combined_df['bug'].unique()).issubset({0, 1}),
    'no_missing_metric_values': combined_df[metric_columns].isna().sum().sum() == 0,
    'all_metric_values_finite': np.isfinite(combined_df[metric_columns].to_numpy(float)).all(),
    'has_dataset_name_column': 'dataset_name' in combined_df.columns,
    'has_source_mapping_columns': {'source_path', 'match_strategy'}.issubset(combined_df.columns),
}
checks

## 7. Single-Project Usage Example

Use this command pattern when you want to preprocess only one dataset with an explicit CSV path and source-code path:

```bash
python scripts/preprocess_promise.py \
  --dataset-name ant-1.6 \
  --csv-path projects/ant/ant-1.6.csv \
  --source-root projects/ant
```

The default multi-project command is still preferred for training on the combined SDP dataset.